# backward-func-lookup — worked example 1: Build BackwardFuncLookup and count its registered keys

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-func-lookup`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

`BackwardFuncLookup` is just a `dict` keyed by the 2-tuple `(forward_fn, arg_position)`. `add_back_func` writes one entry; `get_back_func` reads it back in `O(1)`. Because the key is flat, registering one back fn for several argnums means several independent dict entries, and you can count registrations simply by `len(self.back_funcs)`.

## Worked solution

**Step 1 — the storage.** `__init__` creates `self.back_funcs = {}`. Every registration is one key/value pair, so the dict's length is exactly the number of `(fwd, argnum)` pairs ever inserted (overwrites do not grow it).

**Step 2 — insertion.** `add_back_func(fwd, argnum, back_fn)` does `self.back_funcs[(fwd, argnum)] = back_fn`. The tuple is hashable because `fwd` is a function object (hashable by identity) and `argnum` is an int.

**Step 3 — lookup.** `get_back_func` rebuilds the same tuple key and returns the stored value, raising a clear `KeyError` naming both the function and argnum when absent — that message is what saves you when a registration is missing during a reverse pass.

**Step 4 — counting.** We register two argnums for `t.multiply` and one for `t.log`. That is three distinct keys, so `len(BFL.back_funcs) == 3`. Re-registering `(t.log, 0)` overwrites in place, leaving the count unchanged — proving the count reflects distinct keys, not call count.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={arg_position}).')
        return self.back_funcs[key]


def mul_back0(grad_out, out, x, y):
    return grad_out * y

def mul_back1(grad_out, out, x, y):
    return grad_out * x

def log_back0(grad_out, out, x):
    return grad_out / x

BFL = BackwardFuncLookup()
BFL.add_back_func(t.multiply, 0, mul_back0)
BFL.add_back_func(t.multiply, 1, mul_back1)
BFL.add_back_func(t.log, 0, log_back0)
# Re-registering an existing key overwrites, does NOT add a new entry:
BFL.add_back_func(t.log, 0, log_back0)

print('num registered keys:', len(BFL.back_funcs))
print('multiply argnum0 is mul_back0:', BFL.get_back_func(t.multiply, 0) is mul_back0)